# 02 – Huấn luyện FashionMNIST
Quy trình: DataLoader → mô hình → loss/optimizer → huấn luyện (Forward → Loss → Backward → Optimize) → đánh giá → vẽ Loss → lưu và tải lại mô hình.

Chạy từ thư mục `notebooks/`. Lần đầu `download=True` sẽ tải dữ liệu vào `data/input/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # để import được src.*

import torch
from torch import nn
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter

from src.config import (BATCH_SIZE, LEARNING_RATE, EPOCHS, RANDOM_SEED, CLASS_NAMES,
                        NUM_CLASSES, TENSORBOARD_DIR, TABLES_DIR, CHECKPOINT_DIR,
                        ensure_dirs, set_seed, get_device)
from src.preprocessing.dataset import get_data_loaders
from src.models.neural_network import CNN
from src.models.model_io import save_model, load_model
from src.training.trainer import fit
from src.evaluation.metrics import evaluate, predict_all

ensure_dirs()
set_seed(RANDOM_SEED)
device = get_device()
print('device:', device)

## 1. Dữ liệu
`get_data_loaders` tải FashionMNIST (nếu chưa có), áp dụng transform và chia batch.

In [ ]:
train_loader, test_loader = get_data_loaders(batch_size=BATCH_SIZE)
images, labels = next(iter(train_loader))
print('batch ảnh :', tuple(images.shape))   # kỳ vọng (64, 1, 28, 28)
print('batch nhãn:', tuple(labels.shape))   # kỳ vọng (64,)
print('số mẫu train / test:', len(train_loader.dataset), '/', len(test_loader.dataset))
print('pixel min/max sau Normalize:', images.min().item(), images.max().item())  # kỳ vọng ~[-1, 1]

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, img, lab in zip(axes.flat, images, labels):
    ax.imshow(img.squeeze(), cmap='gray'); ax.set_title(CLASS_NAMES[lab], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Mô hình, loss function, optimizer
Mô hình trả logits (không có Softmax) nên dùng `CrossEntropyLoss`, vì hàm này đã gồm LogSoftmax. Cấu hình đang dùng là `CNN` + Adam.

In [ ]:
model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)
print('số tham số:', sum(p.numel() for p in model.parameters()))

## 3. Huấn luyện
Mỗi epoch: duyệt toàn bộ tập train theo Forward → Loss → Backward → Optimize, rồi đánh giá trên tập test. Log TensorBoard: `tensorboard --logdir data/output/tensorboard`.

In [ ]:
writer = SummaryWriter(str(TENSORBOARD_DIR / 'cnn_run_01'))
history = fit(model, train_loader, test_loader, loss_fn, optimizer,
              epochs=EPOCHS, device=device, writer=writer)
writer.close()

## 4. Đánh giá trên tập test

In [ ]:
final = evaluate(model, test_loader, loss_fn, device)
print(f"test loss = {final['loss']:.4f} | test accuracy = {final['accuracy']:.4f}")

Bảng precision / recall / F1 theo lớp và confusion matrix. Tính trực tiếp bằng tensor nên không cần thêm scikit-learn.

In [ ]:
y_true, y_pred = predict_all(model, test_loader, device)
cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1          # hàng = nhãn thật, cột = nhãn dự đoán

tp = cm.diag().float()
precision = tp / cm.sum(dim=0).clamp(min=1)
recall = tp / cm.sum(dim=1).clamp(min=1)
f1 = 2 * precision * recall / (precision + recall).clamp(min=1e-12)
report = pd.DataFrame({'precision': precision, 'recall': recall, 'f1': f1,
                       'support': cm.sum(dim=1)}, index=CLASS_NAMES).round(4)
report.to_csv(TABLES_DIR / 'class_report.csv')
report

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=90)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set(xlabel='Dự đoán', ylabel='Thực tế', title='Confusion matrix (tập test)')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=7,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

## 5. Loss và Accuracy theo epoch
History được lưu ra CSV để Đức dùng khi vẽ biểu đồ cuối.

In [ ]:
df = pd.DataFrame(history)
df.to_csv(TABLES_DIR / 'training_history.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(df.epoch, df.train_loss, label='train'); ax[0].plot(df.epoch, df.test_loss, label='test')
ax[0].set(title='Loss', xlabel='epoch'); ax[0].legend()
ax[1].plot(df.epoch, df.train_accuracy, label='train'); ax[1].plot(df.epoch, df.test_accuracy, label='test')
ax[1].set(title='Accuracy', xlabel='epoch'); ax[1].legend()
plt.tight_layout(); plt.show()
df

## 6. Lưu mô hình và tải lại
Kiểm tra: mô hình tải lại phải cho đúng accuracy như trước khi lưu.

In [ ]:
ckpt_path = save_model(model)   # mặc định: data/output/checkpoints/fashion_mnist_model.pth
print('đã lưu:', ckpt_path)

reloaded = load_model(CNN(), ckpt_path, device)
after = evaluate(reloaded, test_loader, loss_fn, device)
print('accuracy trước khi lưu :', final['accuracy'])
print('accuracy sau khi tải lại:', after['accuracy'])
assert abs(final['accuracy'] - after['accuracy']) < 1e-9, 'Accuracy lệch sau khi tải lại!'

## Ghi chú cho báo cáo
Điền số liệu thật vào `docs/06_results/evaluation_results.md` sau khi chạy: cấu hình, loss/accuracy cuối, các lớp bị nhầm nhiều (xem confusion matrix) và khoảng cách train–test.